# Create Crafoord Prize Awards (PRIZE PATTERN)

Crafoord Prize laureates from the official Crafoord Prize WordPress REST API.

**Prerequisite:** run `scripts/local/crafoord_prize_to_s3.py` to fetch official `award_winner` API records, verify the official 2015-present amount-rule press release, and upload parquet to S3.

**Source:** `https://www.crafoordprize.se/about-the-prize/laureates/`, backed by `https://www.crafoordprize.se/wp-json/wp/v2/award_winner`; amount-rule source is `https://www.crafoordprize.se/news/the-prize-amount-for-the-crafoord-prize-has-been-increased-to-6-million-swedish-krona/`.

**S3 location:** `s3://openalex-ingest/awards/crafoord_prize/crafoord_prize_laureates.parquet`

**OpenAlex funder:** Royal Swedish Academy of Sciences, `funder_id = 4320320936`, DOI `10.13039/501100001725`; no ROR currently exposed by OpenAlex.

**Awarding-body note:** the official Crafoord Prize site states that the prize is awarded in partnership between the Royal Swedish Academy of Sciences and the Crafoord Foundation in Lund, and that the Academy is responsible for selecting the laureates. OpenAlex has the Royal Swedish Academy of Sciences as a funder; a distinct `Crafoord Prize` funder was not found.

**Schema notes:**
- Prize pattern: the official API already emits one row per laureate.
- `lead_investigator` is the laureate.
- `funding_type = 'prize'`.
- Provenance is `crafoord_prize`.
- For 2015-present rows, `amount` is the official SEK 6,000,000 total prize amount divided by the number of laureates in the same year/category. Pre-2015 amounts are left NULL because this official source only verifies the 2015-present amount rule.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crafoord_prize_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/crafoord_prize/crafoord_prize_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.crafoord_prize_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.crafoord_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.crafoord_prize_raw
LIMIT 10;


In [ ]:
%sql
-- Money-shaped columns from the raw source. 2015-present amounts are derived
-- from the official Crafoord Prize amount-rule press release and stored as strings.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'crafoord_prize_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|fund|award|value|money|prize|sek|krona';


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(source_award_amount) AS rows_with_amount,
    MIN(TRY_CAST(source_award_amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(source_award_amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(source_award_amount AS DOUBLE)) AS avg_amount,
    collect_set(amount_rule_url) AS amount_rule_urls
FROM openalex.awards.crafoord_prize_raw
GROUP BY currency
ORDER BY currency;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(laureate_name) AS has_name,
    COUNT(award_year) AS has_year,
    COUNT(prize_category) AS has_category,
    COUNT(affiliation) AS has_affiliation,
    COUNT(citation) AS has_citation,
    COUNT(source_award_amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.crafoord_prize_raw;


In [ ]:
%sql
SELECT
    award_year,
    prize_category,
    laureate_name,
    affiliation,
    source_award_amount,
    currency,
    award_share_count,
    portion,
    landing_page_url
FROM openalex.awards.crafoord_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, prize_category, laureate_name
LIMIT 50;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320936;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crafoord_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320936
),
normalized AS (
    SELECT
        n.*,
        TRY_CAST(n.award_year AS INT) AS award_year_int,
        TRY_CAST(n.source_award_amount AS DOUBLE) AS amount_double
    FROM openalex.awards.crafoord_prize_raw n
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':crafoord-prize:', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        CONCAT('Crafoord Prize ', TRY_CAST(n.award_year_int AS STRING), ' - ', n.prize_category, ' - ', n.laureate_name) AS display_name,
        NULLIF(n.citation, '') AS description,
        f.funder_id,
        n.funder_award_id,
        n.amount_double AS amount,
        NULLIF(n.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        NULLIF(n.prize_category, '') AS funder_scheme,
        'crafoord_prize' AS provenance,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            NULLIF(n.given_name, '') AS given_name,
            NULLIF(n.family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(n.affiliation, '') AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        NULLIF(n.landing_page_url, '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.laureate_name IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', TRY_CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


## Step 3: Insert Into `openalex_awards_raw` With Priority 69

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crafoord_prize' AND priority = 69;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    69 AS priority
FROM openalex.awards.crafoord_prize_awards;


## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.crafoord_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.crafoord_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.crafoord_prize_awards;


In [ ]:
%sql
SELECT
    funder.display_name AS funder_display_name,
    funder_id,
    COUNT(*) AS rows
FROM openalex.awards.crafoord_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    funder_scheme,
    currency,
    COUNT(*) AS rows,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.crafoord_prize_awards
GROUP BY funder_scheme, currency
ORDER BY funder_scheme, currency;


In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS rows
FROM openalex.awards.crafoord_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Duplicate checks. Both should return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.crafoord_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.crafoord_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    amount,
    currency,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.crafoord_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 20;
